# Team 13 — Feed-Forward Neural Network
## MSE 546: Detecting AI-Generated Text

This notebook implements a **Feed-Forward Neural Network** for the Kaggle competition "LLM - Detect AI Generated Text".

**Approach:** TF-IDF Vectorization → Feed-Forward NN (PyTorch)

We reuse the same dataset, split, and TF-IDF pipeline as the baseline to ensure a **fair comparison**.

## Step 0: Setup

If running in **Google Colab**, run the cell below to upload your data and install PyTorch.  
If running **locally**, skip this cell.

In [ ]:
# ============================================================
# Setup for Google Colab (skip if running locally)
# ============================================================
import os
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally (not in Colab)")

if IN_COLAB:
    from google.colab import files
    if not os.path.exists('train_v4_drcat_01.csv'):
        print("\n" + "="*50)
        print("Please upload 'train_v4_drcat_01.csv'")
        print("(from your DAIGT-V4-TRAIN-DATASEt folder)")
        print("="*50 + "\n")
        uploaded = files.upload()
        print("\nFile uploaded successfully!")
    else:
        print("Data file already exists, skipping upload.")

## Step 1: Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Device selection: Apple MPS > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# Load Data - Same as baseline
# ============================================================
import os

possible_paths = [
    "train_v4_drcat_01.csv",
    "DAIGT-V4-TRAIN-DATASEt/train_v4_drcat_01.csv",
    "../DAIGT-V4-TRAIN-DATASEt/train_v4_drcat_01.csv",
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find data file! Please either:\n"
        "1. Upload 'train_v4_drcat_01.csv' in Colab, or\n"
        "2. Make sure the DAIGT-V4 folder is in the correct location locally"
    )

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

print(f"\nDataset shape: {df.shape}")
print(f"Class Distribution:")
print(df['label'].value_counts())

## Step 2: Prepare Data (Same as Baseline)

We use the **exact same** preprocessing and split to ensure fair comparison:
- 80/20 stratified train/test split
- TF-IDF with 10,000 features, unigrams + bigrams

In [ ]:
# Same split as baseline
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training set: {len(X_train)} essays")
print(f"Test set:     {len(X_test)} essays")

In [ ]:
# Same TF-IDF as baseline
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    stop_words='english'
)

print("Fitting TF-IDF on training data...")
X_train_tfidf = tfidf.fit_transform(X_train)

print("Transforming test data...")
X_test_tfidf = tfidf.transform(X_test)

print(f"\nFeature matrix shape: {X_train_tfidf.shape}")

## Step 3: Convert Data to PyTorch Tensors

PyTorch works with tensors (like NumPy arrays but with GPU support).  
We also create **DataLoaders** which feed data in mini-batches during training  
(this is the "SGD with minibatch" from Lecture 6).

In [ ]:
# Convert sparse TF-IDF matrices to dense tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
y_train_tensor = torch.FloatTensor(y_train.values)

X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())
y_test_tensor = torch.FloatTensor(y_test.values)

# Create DataLoaders for mini-batch training
BATCH_SIZE = 64

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)} (batch size = {BATCH_SIZE})")
print(f"Test batches:     {len(test_loader)}")

## Step 4: Define the Neural Network

Our architecture (from Lecture 6 notation):

```
Input (10,000) → Hidden Layer 1 (512, ReLU) → Dropout
              → Hidden Layer 2 (256, ReLU) → Dropout
              → Hidden Layer 3 (64, ReLU)  → Dropout
              → Output (1, Sigmoid)
```

In Lecture 6 terms:
- **L = 4** layers (3 hidden + 1 output)
- **W₁** ∈ R^(512×10000), **h₁** = ReLU
- **W₂** ∈ R^(256×512), **h₂** = ReLU
- **W₃** ∈ R^(64×256), **h₃** = ReLU
- **W₄** ∈ R^(1×64), **h₄** = Sigmoid
- **Dropout** randomly zeros 30% of neurons during training to prevent overfitting

This architecture was selected based on our hyperparameter exploration (Step 11), where it achieved the highest AUC among all tested configurations.

In [ ]:
class AITextDetector(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.3):
        super(AITextDetector, self).__init__()
        self.network = nn.Sequential(
            # Hidden layer 1: 10,000 → 512
            nn.Linear(input_dim, 512),
            nn.ReLU(),                    # Activation function (Lecture 6)
            nn.Dropout(dropout_rate),      # Regularization
            
            # Hidden layer 2: 512 → 256
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Hidden layer 3: 256 → 64
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # Output layer: 64 → 1
            nn.Linear(64, 1),
            nn.Sigmoid()                  # Output probability [0, 1]
        )
    
    def forward(self, x):
        return self.network(x).squeeze()

# Create model
INPUT_DIM = X_train_tfidf.shape[1]  # 10,000
model = AITextDetector(INPUT_DIM).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 5: Training Setup

- **Loss function**: Binary Cross-Entropy — this is the logistic loss from Lecture 4:  
  `ℓ(z) = log(1 + exp(-z))`
- **Optimizer**: Adam — a variant of SGD with adaptive learning rates and momentum (Lecture 6)
- **Early stopping**: Stop training if the model stops improving on a validation set

In [ ]:
# Loss and optimizer
criterion = nn.BCELoss()                                    # Binary Cross-Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)        # Adam optimizer

# Training configuration
NUM_EPOCHS = 20
PATIENCE = 3  # Stop if no improvement for 3 epochs

print(f"Loss function: Binary Cross-Entropy")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Early stopping patience: {PATIENCE}")

## Step 6: Train the Model

This is the **Backpropagation algorithm from Lecture 6** (slide 19):

1. Pick a mini-batch of data
2. **Forward propagation**: compute output layer by layer
3. **Backward propagation**: compute gradients via chain rule
4. **Update weights**: W ← W − η × gradient

In [ ]:
# Track metrics across epochs
train_losses = []
test_losses = []
test_aucs = []

best_auc = 0.0
best_epoch = 0
patience_counter = 0
best_model_state = None

print("Training started...")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    # ---- Training phase ----
    model.train()  # Enable dropout
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward propagation (Lecture 6: compute a_l and o_l for each layer)
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward propagation (Lecture 6: compute gradients via chain rule)
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute gradients
        
        # Update weights (Lecture 6: W_l ← W_l − η * dE/dW_l)
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_train_loss = epoch_loss / num_batches
    train_losses.append(avg_train_loss)
    
    # ---- Evaluation phase ----
    model.eval()  # Disable dropout
    test_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():  # No gradient computation needed for evaluation
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item()
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(batch_y.cpu().numpy())
    
    avg_test_loss = test_loss / len(test_loader)
    test_losses.append(avg_test_loss)
    
    # Calculate AUC
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    epoch_auc = roc_auc_score(all_labels, all_preds)
    test_aucs.append(epoch_auc)
    
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Test Loss: {avg_test_loss:.4f} | "
          f"Test AUC: {epoch_auc:.4f}", end="")
    
    # Early stopping check
    if epoch_auc > best_auc:
        best_auc = epoch_auc
        best_epoch = epoch + 1
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        print(" ★ Best")
    else:
        patience_counter += 1
        print()
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)")
            break

print("=" * 70)
print(f"Best model: Epoch {best_epoch} with AUC = {best_auc:.4f}")

# Restore best model
model.load_state_dict(best_model_state)
print("Best model restored.")

## Step 7: Training Curves

Visualize how the model learned over time. A good training curve shows:
- Both losses decreasing together (model is learning)
- Test loss not rising while train loss falls (no overfitting)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(range(1, len(train_losses)+1), train_losses, 'b-o', label='Train Loss', markersize=5)
axes[0].plot(range(1, len(test_losses)+1), test_losses, 'r-o', label='Test Loss', markersize=5)
axes[0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Binary Cross-Entropy Loss', fontsize=12)
axes[0].set_title('Training and Test Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# AUC curve
axes[1].plot(range(1, len(test_aucs)+1), test_aucs, 'g-o', label='Test AUC', markersize=5)
axes[1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_epoch})')
axes[1].axhline(y=0.9924, color='orange', linestyle='--', alpha=0.7, label='Baseline AUC (0.9924)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('ROC-AUC Score', fontsize=12)
axes[1].set_title('Test AUC Over Epochs', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: nn_training_curves.png")

## Step 8: Evaluate Final Model Performance

In [ ]:
# Generate predictions with best model
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        all_preds.append(outputs.cpu().numpy())
        all_labels.append(batch_y.numpy())

y_pred_proba = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)
y_pred = (y_pred_proba >= 0.5).astype(int)

# Calculate metrics
nn_roc_auc = roc_auc_score(y_true, y_pred_proba)
nn_precision = precision_score(y_true, y_pred)
nn_recall = recall_score(y_true, y_pred)
nn_f1 = f1_score(y_true, y_pred)

print("=" * 50)
print("NEURAL NETWORK MODEL PERFORMANCE")
print("=" * 50)
print(f"\nROC-AUC Score:  {nn_roc_auc:.4f}")
print(f"Precision:      {nn_precision:.4f}")
print(f"Recall:         {nn_recall:.4f}")
print(f"F1-Score:       {nn_f1:.4f}")
print("=" * 50)

print("\nDetailed Classification Report:")
print("-" * 50)
print(classification_report(y_true, y_pred, target_names=['Human (0)', 'AI (1)']))

## Step 9: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Human (0)', 'AI (1)'],
            yticklabels=['Human (0)', 'AI (1)'],
            annot_kws={'size': 14})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix — Feed-Forward Neural Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('nn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives (Human→Human):  {tn}")
print(f"False Positives (Human→AI):     {fp}")
print(f"False Negatives (AI→Human):     {fn}")
print(f"True Positives (AI→AI):         {tp}")

## Step 10: ROC Curve — Baseline vs Neural Network

In [ ]:
# ---- Retrain the baseline for direct comparison ----
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0, solver='lbfgs')
baseline_model.fit(X_train_tfidf, y_train)
baseline_proba = baseline_model.predict_proba(X_test_tfidf)[:, 1]
baseline_auc = roc_auc_score(y_test, baseline_proba)

# ROC curves
fpr_baseline, tpr_baseline, _ = roc_curve(y_test, baseline_proba)
fpr_nn, tpr_nn, _ = roc_curve(y_true, y_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_baseline, tpr_baseline, 'b-', lw=2, label=f'Baseline — LR (AUC = {baseline_auc:.4f})')
plt.plot(fpr_nn, tpr_nn, 'g-', lw=2, label=f'Neural Network (AUC = {nn_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Baseline vs Neural Network', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('nn_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: nn_roc_comparison.png")

## Step 11: Hyperparameter Exploration

We test different architectures to show we explored design choices.  
This demonstrates understanding and helps justify our final architecture.

In [ ]:
# Define architectures to compare
architectures = {
    '1 Layer (256)': [256],
    '1 Layer (512)': [512],
    '2 Layers (512→128)': [512, 128],
    '2 Layers (256→64)': [256, 64],
    '3 Layers (512→256→64)': [512, 256, 64],
}

def build_model(input_dim, hidden_layers, dropout=0.3):
    """Build a feed-forward NN with given hidden layer sizes."""
    layers = []
    prev_dim = input_dim
    for hidden_dim in hidden_layers:
        layers.extend([
            nn.Linear(prev_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        ])
        prev_dim = hidden_dim
    layers.extend([nn.Linear(prev_dim, 1), nn.Sigmoid()])
    return nn.Sequential(*layers)

def train_and_evaluate(model, train_loader, test_loader, device, epochs=10):
    """Train a model and return its best test AUC."""
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    best_auc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        model.eval()
        preds = []
        labels = []
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                outputs = model(batch_X).squeeze()
                preds.append(outputs.cpu().numpy())
                labels.append(batch_y.numpy())
        
        auc = roc_auc_score(np.concatenate(labels), np.concatenate(preds))
        best_auc = max(best_auc, auc)
    
    return best_auc

# Run experiments
print("Comparing architectures (10 epochs each)...")
print("=" * 55)

results = {}
for name, hidden_layers in architectures.items():
    torch.manual_seed(RANDOM_STATE)
    m = build_model(INPUT_DIM, hidden_layers)
    n_params = sum(p.numel() for p in m.parameters())
    auc = train_and_evaluate(m, train_loader, test_loader, device, epochs=10)
    results[name] = {'auc': auc, 'params': n_params}
    print(f"{name:30s} | Params: {n_params:>10,} | Best AUC: {auc:.4f}")

print("=" * 55)

In [ ]:
# Visualize architecture comparison
names = list(results.keys())
aucs = [results[n]['auc'] for n in names]
params = [results[n]['params'] for n in names]

fig, ax1 = plt.subplots(figsize=(12, 6))

x = range(len(names))
bars = ax1.bar(x, aucs, color='steelblue', alpha=0.8, width=0.5)
ax1.axhline(y=0.9924, color='orange', linestyle='--', linewidth=2, label='Baseline AUC (0.9924)')
ax1.set_ylabel('ROC-AUC Score', fontsize=12)
ax1.set_title('Architecture Comparison — ROC-AUC', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=25, ha='right', fontsize=10)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Add AUC values on bars
for bar, auc in zip(bars, aucs):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
             f'{auc:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('nn_architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: nn_architecture_comparison.png")

## Step 12: Final Comparison — Baseline vs Neural Network

In [ ]:
# Baseline metrics (from baseline notebook)
baseline_pred = baseline_model.predict(X_test_tfidf)
baseline_metrics = {
    'ROC-AUC': roc_auc_score(y_test, baseline_proba),
    'Precision': precision_score(y_test, baseline_pred),
    'Recall': recall_score(y_test, baseline_pred),
    'F1-Score': f1_score(y_test, baseline_pred),
}

nn_metrics = {
    'ROC-AUC': nn_roc_auc,
    'Precision': nn_precision,
    'Recall': nn_recall,
    'F1-Score': nn_f1,
}

# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': list(baseline_metrics.keys()),
    'Baseline (TF-IDF + LR)': list(baseline_metrics.values()),
    'Neural Network': list(nn_metrics.values()),
})

comparison_df['Difference'] = comparison_df['Neural Network'] - comparison_df['Baseline (TF-IDF + LR)']
comparison_df['Change'] = comparison_df['Difference'].apply(
    lambda x: f'+{x:.4f}' if x >= 0 else f'{x:.4f}'
)

print("=" * 65)
print("FINAL COMPARISON: BASELINE vs NEURAL NETWORK")
print("=" * 65)
print(comparison_df[['Metric', 'Baseline (TF-IDF + LR)', 'Neural Network', 'Change']].to_string(index=False))
print("=" * 65)

In [ ]:
# Side-by-side bar chart
metrics = list(baseline_metrics.keys())
baseline_vals = list(baseline_metrics.values())
nn_vals = list(nn_metrics.values())

x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline (TF-IDF + LR)', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, nn_vals, width, label='Neural Network', color='forestgreen', alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('nn_vs_baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: nn_vs_baseline_comparison.png")

## Step 13: Summary for Report

In [ ]:
print("=" * 60)
print("SUMMARY FOR SLIDES")
print("=" * 60)
print(f"""
Neural Network Model: Feed-Forward NN (PyTorch)

Architecture (best from hyperparameter search):
  - Input:   {INPUT_DIM:,} features (TF-IDF)
  - Layer 1: 512 neurons, ReLU, Dropout(0.3)
  - Layer 2: 256 neurons, ReLU, Dropout(0.3)
  - Layer 3: 64 neurons, ReLU, Dropout(0.3)
  - Output:  1 neuron, Sigmoid
  - Total parameters: {total_params:,}

Training:
  - Loss: Binary Cross-Entropy
  - Optimizer: Adam (lr=0.001)
  - Batch size: {BATCH_SIZE}
  - Best epoch: {best_epoch}
  - Early stopping patience: {PATIENCE}

Performance:
  - ROC-AUC:   {nn_roc_auc:.4f}
  - Precision:  {nn_precision:.4f}
  - Recall:     {nn_recall:.4f}
  - F1-Score:   {nn_f1:.4f}

Baseline Comparison:
  - Baseline AUC: {baseline_metrics['ROC-AUC']:.4f}
  - NN AUC:       {nn_roc_auc:.4f}
  - Difference:   {nn_roc_auc - baseline_metrics['ROC-AUC']:+.4f}

Saved Plots:
  - nn_training_curves.png
  - nn_confusion_matrix.png
  - nn_roc_comparison.png
  - nn_architecture_comparison.png
  - nn_vs_baseline_comparison.png
""")
print("=" * 60)